# Overlap flow: how neighbouring tiles sit relative to each other, and the stitched result

For each pair of overlapping tiles, the contacts both tiles see are compared: B − A, at nominal
placement. The mask error is identical in both images, so this difference is pure SEM placement
and measurement error.

1. **Robust translation** (`fit_overlap`): least-squares shift; contacts whose residual exceeds
   `OUTLIER_FACTOR` × the median residual are outliers and left out. RMS is shown for inliers
   (the noise floor) and for all points.
2. **Image registration** (`image_overlap_shift`): the shift from the overlap pixels, independent
   of contact detection. It should agree with step 1 within about ±0.3 nm.
3. **Translation + rotation** (`fit_overlap(..., rotation=True)`): rotation shows up as a linear
   trend of the across-strip residual along the strip.
4. **Stitching**: one translation correction per tile from all pairs (`solve_tile_shifts`), then
   every pair's differences before/after correction, and the stitched mosaic before/after.

Positions are in µm, differences in nm, rotations in µrad. Raw B − A differences are drawn
magnified by `ARROW_SCALE`; residuals after a correction (much smaller) by `RESIDUAL_SCALE`.
Each plot's title states its magnification and its key arrow shows 1 nm. Plots are interactive
(zoom/pan).

In [ ]:
DATA_DIR = r"A:\Stitch_monitor\In_cell_without_PTM_20260914\260910_1143_Norm"
TILES = list(range(10))   # which tiles to use (indices in read_tile_index order)
MOSAIC_TILES = TILES[:4]  # tiles shown in the stitched mosaic (a few, to keep the image small)
REFINE_EDGES = True       # refine Otsu contours to the maximum intensity gradient
SEARCH_PX = 5.0           # refinement searches this many pixels in/out along the contour normal
MAX_MATCH_NM = 25.0       # pair contacts closer than this: the largest B - A stage offset (must be < pitch / 2)
OUTLIER_FACTOR = 3.0      # outlier if residual > this x the median residual
MIN_MATCHED = 5           # skip pairs with fewer matched contacts (e.g. corner-only overlaps)
PAIR = 0                  # which pair (row of the pair table) to look at in detail
ARROW_SCALE = 50          # raw B - A difference arrows are drawn this many times longer
RESIDUAL_SCALE = 500      # residual arrows (after a correction) are drawn this many times longer
ZOOM_UM = 0.6             # size of the zoomed mosaic view around an overlap strip (µm)

In [ ]:
%matplotlib widget
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from affine_ransac.features.contact import detect_contacts
from affine_ransac.features.edges import refine_edges
from affine_ransac.geometry.frames import pixel_to_tile_nm
from affine_ransac.io.metadata import read_tile_index
from affine_ransac.io.sem_image import load_sem_image
from affine_ransac.mosaic import build_mosaic
from affine_ransac.overlap import match_overlap, overlapping_pairs
from affine_ransac.overlap_fit import fit_overlap
from affine_ransac.overlap_image import crop_shift_px, image_overlap_shift, overlap_crops
from affine_ransac.stitching import solve_tile_shifts
from overlap_plots import (plot_all_pairs, plot_image_alignment, plot_mosaic, plot_rotation_fit,
                           plot_stitch_residuals, plot_translation_fit)

## Load the tiles and detect their contacts

In [ ]:
all_tiles = read_tile_index(DATA_DIR)
tiles = [all_tiles[i] for i in TILES]
tile_ids = [t.tile_id for t in tiles]
centers = np.array([(t.center_x_nm, t.center_y_nm) for t in tiles])
fovs = np.array([(t.fov_x_nm, t.fov_y_nm) for t in tiles])
pixel_size_nm = tiles[0].fov_x_nm / tiles[0].image_width_px

start = time.perf_counter()
images, found, contacts = [], [], []   # per tile; contacts = SEM centres in mask nm, nominal placement
for tile, center in zip(tiles, centers):
    image = load_sem_image(tile.image_path)
    detected = detect_contacts(image)
    if REFINE_EDGES:
        detected = refine_edges(image, detected, search_px=SEARCH_PX)
    images.append(image)
    found.append(detected)
    contacts.append(pixel_to_tile_nm(detected.centers, image.shape, pixel_size_nm) + center)
print(f"{len(tiles)} tiles, {sum(len(c) for c in contacts)} contacts, {time.perf_counter() - start:.1f} s")

## Overlapping pairs

In [ ]:
pairs = []   # (i, j, box, matched A points, matched B points)
for i, j, box in overlapping_pairs(centers, fovs):
    ia, ib = match_overlap(contacts[i], contacts[j], box, MAX_MATCH_NM)
    if len(ia) >= MIN_MATCHED:
        pairs.append((i, j, box, contacts[i][ia], contacts[j][ib]))

pd.DataFrame([{
    "tile_a": tile_ids[i], "tile_b": tile_ids[j],
    "overlap_w_um": round((box[1] - box[0]) / 1000, 3), "overlap_h_um": round((box[3] - box[2]) / 1000, 3),
    "matched": len(a),
} for i, j, box, a, b in pairs])

## One pair in detail (`PAIR`)

### Step 1: robust translation

Top: each contact's B − A difference. Middle: what is left after removing the fitted shift
(note the larger magnification). Right: every contact's difference as a point. A tight cloud =
noise; points outside the dashed circle = outliers.

In [ ]:
i, j, box, a, b = pairs[PAIR]
fit_t = fit_overlap(a, b, outlier_factor=OUTLIER_FACTOR)
print(f"Pair {PAIR}: {tile_ids[i]} -> {tile_ids[j]}, {len(a)} matched contacts, {(~fit_t.inliers).sum()} outliers")
print(f"Shift B - A: x {fit_t.shift[0]:+.2f} nm, y {fit_t.shift[1]:+.2f} nm")
print(f"Residual RMS: {fit_t.rms():.2f} nm (inliers), {fit_t.rms(include_outliers=True):.2f} nm (all points)")
plot_translation_fit(a, b, fit_t, ARROW_SCALE, RESIDUAL_SCALE, OUTLIER_FACTOR)

### Step 2: image registration of the overlap pixels

Tile A in magenta, tile B in green; where they line up they look grey. Top: nominal placement.
Bottom: B moved back by the measured image shift. Right: the contact differences with both shift
estimates.

In [ ]:
image_shift, match_error = image_overlap_shift(images[i], centers[i], images[j], centers[j], pixel_size_nm, box)
crop_a, crop_b, _ = overlap_crops(images[i], centers[i], images[j], centers[j], pixel_size_nm, box)
move_px, _ = crop_shift_px(crop_a, crop_b)
print(f"Image shift B - A:   x {image_shift[0]:+.2f} nm, y {image_shift[1]:+.2f} nm (match error {match_error:.2f}, lower is better)")
print(f"Contact shift B - A: x {fit_t.shift[0]:+.2f} nm, y {fit_t.shift[1]:+.2f} nm")
print(f"Image - contacts:    x {image_shift[0] - fit_t.shift[0]:+.2f} nm, y {image_shift[1] - fit_t.shift[1]:+.2f} nm")
plot_image_alignment(crop_a, crop_b, move_px, box, b - a, fit_t.inliers, fit_t.shift, image_shift)

### Step 3: translation + rotation

Top two: residuals after translation only and after translation + rotation, same scale.
Bottom: after translation only, the residual across the strip vs position along it. A rotation
makes these points follow the straight line; random scatter around 0 means no rotation.

In [ ]:
fit_r = fit_overlap(a, b, rotation=True, outlier_factor=OUTLIER_FACTOR)
print(f"Rotation: {fit_r.rotation * 1e6:+.1f} urad")
print(f"Residual RMS (inliers): translation {fit_t.rms():.2f} nm -> translation + rotation {fit_r.rms():.2f} nm")
plot_rotation_fit(a, fit_t, fit_r, RESIDUAL_SCALE)

## Steps 1–3 for all pairs

Top left: does rotation lower the RMS? Top right: are the rotations consistent? Bottom left: do
image and contact shifts agree? Bottom right: matched contacts and outliers per pair. Points are
labelled with the pair number (row of the table below).

In [ ]:
rows, pair_fits = [], []
for i, j, box, a, b in pairs:
    fit_t = fit_overlap(a, b, outlier_factor=OUTLIER_FACTOR)
    fit_r = fit_overlap(a, b, rotation=True, outlier_factor=OUTLIER_FACTOR)
    image_shift, match_error = image_overlap_shift(images[i], centers[i], images[j], centers[j], pixel_size_nm, box)
    pair_fits.append(fit_t)
    rows.append({
        "tile_a": tile_ids[i], "tile_b": tile_ids[j], "matched": len(a), "outliers": int((~fit_t.inliers).sum()),
        "shift_x_nm": fit_t.shift[0], "shift_y_nm": fit_t.shift[1],
        "rms_translation_nm": fit_t.rms(), "rms_rigid_nm": fit_r.rms(), "rotation_urad": fit_r.rotation * 1e6,
        "image_minus_contact_x_nm": image_shift[0] - fit_t.shift[0],
        "image_minus_contact_y_nm": image_shift[1] - fit_t.shift[1], "match_error": match_error,
    })
summary = pd.DataFrame(rows)
plot_all_pairs(summary)
summary.round(2)

## Step 4: stitching correction

One translation per tile, from all pairs' step-1 shifts (weighted by their number of inliers),
averaging zero. Then every pair's B − A differences at their mask positions, before and after
applying the corrections.

In [ ]:
corrections = solve_tile_shifts(len(tiles), [(i, j) for i, j, *_ in pairs], [f.shift for f in pair_fits],
                                weights=[f.inliers.sum() for f in pair_fits])
display(pd.DataFrame({"tile": tile_ids, "correction_x_nm": corrections[:, 0], "correction_y_nm": corrections[:, 1]}).round(2))

pair_data = [(i, j, a, b, fit.inliers) for (i, j, box, a, b), fit in zip(pairs, pair_fits)]
plot_stitch_residuals(centers, fovs, tile_ids, pair_data, corrections, ARROW_SCALE, RESIDUAL_SCALE)

## Stitched mosaic before / after correction (`MOSAIC_TILES`)

Overlapping pixels are averaged, so a misplacement shows up as doubled ("ghost") edges in the
overlap strips (dashed yellow). Each tile's contours and contact centres are drawn in its own
colour; after correction they should coincide in the overlaps. Top: the whole mosaic. Bottom:
`ZOOM_UM` around the middle of the first overlap strip.

In [ ]:
shown = [TILES.index(t) for t in MOSAIC_TILES]
shown_boxes = [box for i, j, box, *_ in pairs if i in shown and j in shown]
x_min, x_max, y_min, y_max = shown_boxes[0]
zoom_center = np.array([(x_min + x_max) / 2, (y_min + y_max) / 2]) / 1000  # µm

fig, axes = plt.subplots(2, 2, figsize=(14, 14), layout="constrained")
for column, (shift, name) in enumerate([(np.zeros_like(corrections), "Before correction"), (corrections, "After correction")]):
    placed = centers + shift
    mosaic, extent = build_mosaic([images[k] for k in shown], placed[shown], pixel_size_nm)
    contours = [[pixel_to_tile_nm(c, images[k].shape, pixel_size_nm) + placed[k] for c in found[k].contours] for k in shown]
    points = [contacts[k] + shift[k] for k in shown]
    for row in (0, 1):
        plot_mosaic(axes[row, column], mosaic, extent, contours, points, shown_boxes,
                    f"{name}: tiles {[tile_ids[k] for k in shown]}" if row == 0 else f"{name}: zoom on an overlap strip")
    axes[1, column].set_xlim(zoom_center[0] - ZOOM_UM / 2, zoom_center[0] + ZOOM_UM / 2)
    axes[1, column].set_ylim(zoom_center[1] - ZOOM_UM / 2, zoom_center[1] + ZOOM_UM / 2)
plt.show()